# Pruebas comparativas — Qdrant
Usa las mismas 20 frases de tickets bancarios, 5 preguntas, modelo, semilla (42), `N=20000` y parámetros HNSW (`M=16`, `ef_construction=100`, `ef_search=64`) que pgvector y Chroma. Guarda el resultado en `resultados_qdrant.txt`.


In [1]:
import os
import statistics
import time
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")

import numpy as np
from sentence_transformers import SentenceTransformer

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

REPETICIONES = 10
N_BENCH = 20000
SEMILLA = 42
M, EF_CONSTRUCTION, EF_SEARCH = 16, 100, 64

documentos = [
    "El cliente reporta que su tarjeta de débito fue rechazada en un cajero automático",
    "Solicitud de aumento de línea de crédito para cuenta empresarial",
    "El cliente no reconoce un cargo de $150 en su estado de cuenta",
    "Consulta sobre requisitos para abrir una cuenta de ahorros infantil",
    "Reclamo por doble cobro en transferencia interbancaria",
    "Solicitud de copia de estado de cuenta de los últimos 6 meses",
    "El cajero automático retuvo la tarjeta del cliente",
    "Consulta sobre tasa de interés para préstamo hipotecario",
    "Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito",
    "Solicitud para cambiar el número de teléfono asociado a la cuenta",
    "El cliente reporta una transacción que no ha realizado con su tarjeta de crédito",
    "No está funcionando las transferencias ACH a las 08:00 am",
    "El usuario de banca en línea fue bloqueado y no puede entrar",
    "La app móvil se encuentra fuera de servicio",
    "El cliente solicita un préstam por consolidación de deudas",
    "La empresa ABC quiere cancelar su cuenta de cheques y abrir una nueva",
    "Se han incrementado las solicitudes de tarjeta de crédto para clientes con historial crediticio limitado",
    "Consulta para cancelar una tarjeta de crédito que ha sido robada",
    "El usuario de la app no puede ver el saldo de la tarjeta de crédito",
    "Solicitud para préstamo de pequeña empresa para capital de trabajo",
]

categorias = [
    "tarjetas", "prestamos", "fraude", "cuentas", "transferencias",
    "cuentas", "tarjetas", "prestamos", "fraude", "cuentas",
    "fraude", "canales_digitales", "canales_digitales", "canales_digitales", "prestamos",
    "cuentas", "tarjetas", "tarjetas", "canales_digitales", "prestamos",
]

preguntas_prueba = [
    ("alguien uso mi tarjeta sin permiso", "Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito"),
    ("me quedé sin poder entrar a la banca en línea", "El usuario de banca en línea fue bloqueado y no puede entrar"),
    ("quiero un crédito para mi negocio", "Solicitud para préstamo de pequeña empresa para capital de trabajo"),
    ("el cajero se tragó mi tarjeta", "El cajero automático retuvo la tarjeta del cliente"),
    ("necesito mis estados de cuenta anteriores", "Solicitud de copia de estado de cuenta de los últimos 6 meses"),
]

lineas = []
def log(msg=""):
    print(msg)
    lineas.append(str(msg))

log("Cargando modelo de embeddings...")
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embedding_docs = modelo.encode(documentos).astype("float32")
DIM = embedding_docs.shape[1]
log(f"Forma de los embeddings: {embedding_docs.shape}")

SALIDA_TXT = RAIZ / "resultados_qdrant.txt"


Cargando modelo de embeddings...
Forma de los embeddings: (20, 384)


In [2]:
from qdrant_client import QdrantClient, models

client = QdrantClient(host="localhost", port=6333, timeout=300)
version = client.info().version
log(f"Base: Qdrant ({version}) | HNSW M={M}, ef_construction={EF_CONSTRUCTION}, ef_search={EF_SEARCH}")

def crear_coleccion(nombre):
    if client.collection_exists(nombre):
        client.delete_collection(nombre)
    client.create_collection(
        nombre,
        vectors_config=models.VectorParams(size=DIM, distance=models.Distance.COSINE),
        hnsw_config=models.HnswConfigDiff(m=M, ef_construct=EF_CONSTRUCTION, full_scan_threshold=10),
        optimizers_config=models.OptimizersConfigDiff(indexing_threshold=10),
    )

def esperar_indice(nombre):
    limite = time.perf_counter() + 300
    while time.perf_counter() < limite:
        info = client.get_collection(nombre)
        if info.status == models.CollectionStatus.GREEN and (info.indexed_vectors_count or 0) >= info.points_count:
            return
        time.sleep(0.2)

COL = "tickets_banco"
crear_coleccion(COL)
client.create_payload_index(COL, field_name="categoria", field_schema="keyword")

t0 = time.perf_counter()
client.upsert(
    COL,
    wait=True,
    points=[
        models.PointStruct(id=i, vector=emb.tolist(), payload={"texto": doc, "categoria": cat})
        for i, (doc, cat, emb) in enumerate(zip(documentos, categorias, embedding_docs))
    ],
)
t_carga = time.perf_counter() - t0
log(f"\n[1] Carga de {client.count(COL).count} frases: {t_carga:.3f} s")


Base: Qdrant (1.20.0) | HNSW M=16, ef_construction=100, ef_search=64

[1] Carga de 20 frases: 0.015 s


In [3]:
def buscar(consulta_vec, k, categoria=None):
    filtro = None
    if categoria:
        filtro = models.Filter(must=[models.FieldCondition(key="categoria", match=models.MatchValue(value=categoria))])
    res = client.query_points(
        COL,
        query=consulta_vec,
        limit=k,
        query_filter=filtro,
        search_params=models.SearchParams(hnsw_ef=EF_SEARCH),
    )
    return [p.payload["texto"] for p in res.points], [float(p.score) for p in res.points]

vectores_q = [modelo.encode(p).astype("float32").tolist() for p, _ in preguntas_prueba]
buscar(vectores_q[0], 3)  # calentamiento

log("\n[2] Aciertos y tiempos (solo la consulta a Qdrant)")
resumen = {}
for k in (3, 5, 10):
    aciertos, tiempos = 0, []
    for (pregunta, esperado), v in zip(preguntas_prueba, vectores_q):
        textos, _ = buscar(v, k)
        if esperado in textos:
            aciertos += 1
        for _ in range(REPETICIONES):
            t0 = time.perf_counter()
            buscar(v, k)
            tiempos.append((time.perf_counter() - t0) * 1000)
    resumen[k] = (aciertos, min(tiempos), max(tiempos), statistics.mean(tiempos))
    log(f"  K={k:<2} aciertos {aciertos}/{len(preguntas_prueba)} | "
        f"min {min(tiempos):.1f} | max {max(tiempos):.1f} | "
        f"prom {statistics.mean(tiempos):.1f} ms")

log("\n  Detalle (K=3):")
for (pregunta, esperado), v in zip(preguntas_prueba, vectores_q):
    textos, scores = buscar(v, 3)
    log(("  OK  " if esperado in textos else "  FALLO") + " " + pregunta)
    if esperado not in textos:
        log("     Esperado: " + esperado)
        for t, s in zip(textos, scores):
            log(f"     {s:.3f} | {t}")



[2] Aciertos y tiempos (solo la consulta a Qdrant)
  K=3  aciertos 4/5 | min 4.8 | max 36.4 | prom 18.0 ms
  K=5  aciertos 4/5 | min 6.0 | max 31.7 | prom 19.3 ms
  K=10 aciertos 5/5 | min 5.1 | max 26.4 | prom 15.4 ms

  Detalle (K=3):
  FALLO alguien uso mi tarjeta sin permiso
     Esperado: Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito
     0.608 | El cliente reporta que su tarjeta de débito fue rechazada en un cajero automático
     0.602 | El cajero automático retuvo la tarjeta del cliente
     0.586 | Consulta para cancelar una tarjeta de crédito que ha sido robada
  OK   me quedé sin poder entrar a la banca en línea
  OK   quiero un crédito para mi negocio
  OK   el cajero se tragó mi tarjeta
  OK   necesito mis estados de cuenta anteriores


In [4]:
log("\n[3] Filtro por metadatos")
v = modelo.encode("alguien uso mi tarjeta sin permiso").astype("float32").tolist()
log("  SIN filtro:")
for t, s in zip(*buscar(v, 3)):
    log(f"    {s:.3f} | {t}")
log("  CON filtro categoria=fraude:")
for t, s in zip(*buscar(v, 3, categoria="fraude")):
    log(f"    {s:.3f} | {t}")



[3] Filtro por metadatos
  SIN filtro:
    0.608 | El cliente reporta que su tarjeta de débito fue rechazada en un cajero automático
    0.602 | El cajero automático retuvo la tarjeta del cliente
    0.586 | Consulta para cancelar una tarjeta de crédito que ha sido robada
  CON filtro categoria=fraude:
    0.576 | El cliente reporta una transacción que no ha realizado con su tarjeta de crédito
    0.491 | Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito
    0.411 | El cliente no reconoce un cargo de $150 en su estado de cuenta


In [5]:
BENCH = "bench"
crear_coleccion(BENCH)

rng = np.random.default_rng(SEMILLA)
datos = rng.normal(size=(N_BENCH, DIM)).astype("float32")

t0 = time.perf_counter()
for i in range(0, N_BENCH, 1000):
    fin = min(i + 1000, N_BENCH)
    client.upsert(
        BENCH,
        wait=True,
        points=[models.PointStruct(id=j, vector=datos[j].tolist()) for j in range(i, fin)],
    )
t_bench = time.perf_counter() - t0

t0 = time.perf_counter()
esperar_indice(BENCH)
t_indice = time.perf_counter() - t0
log(f"\n[4] Carga de {N_BENCH} vectores: {t_bench:.2f} s | Índice HNSW: {t_indice:.2f} s (total {t_bench + t_indice:.2f} s)")

consulta = rng.normal(size=DIM).astype("float32").tolist()
tiempos = []
for _ in range(100):
    t0 = time.perf_counter()
    client.query_points(BENCH, query=consulta, limit=10, search_params=models.SearchParams(hnsw_ef=EF_SEARCH))
    tiempos.append((time.perf_counter() - t0) * 1000)
log(f"    Latencia con {N_BENCH} vectores: promedio {np.mean(tiempos):.1f} ms | p95 {np.percentile(tiempos, 95):.1f} ms")



[4] Carga de 20000 vectores: 4.91 s | Índice HNSW: 3.91 s (total 8.82 s)
    Latencia con 20000 vectores: promedio 17.8 ms | p95 32.0 ms


In [6]:
norm = datos / np.linalg.norm(datos, axis=1, keepdims=True)
aciertos, total = 0, 0
for _ in range(50):
    q = rng.normal(size=DIM).astype("float32")
    aprox = {int(p.id) for p in client.query_points(
        BENCH, query=q.tolist(), limit=10, search_params=models.SearchParams(hnsw_ef=EF_SEARCH)
    ).points}
    sims = norm @ (q / np.linalg.norm(q))
    exacta = set(np.argsort(-sims)[:10].tolist())
    aciertos += len(aprox & exacta)
    total += 10
recall = aciertos / total
log(f"[5] Recall@10 (HNSW vs exacta): {recall:.2%}")

log("\n================ RESUMEN QDRANT ================")
log(f"Carga 20 frases:        {t_carga:.3f} s")
for k, (a, mn, mx, pr) in resumen.items():
    log(f"K={k:<2} aciertos {a}/5 | tiempo min/max/prom: {mn:.1f} / {mx:.1f} / {pr:.1f} ms")
log(f"Carga {N_BENCH} vectores: {t_bench:.2f} s (+ índice {t_indice:.2f} s)")
log(f"Latencia {N_BENCH} (prom): {np.mean(tiempos):.1f} ms | p95 {np.percentile(tiempos, 95):.1f} ms")
log(f"Recall@10:              {recall:.2%}")

SALIDA_TXT.write_text("\n".join(lineas) + "\n", encoding="utf-8")
print(f"\nGuardado en {SALIDA_TXT}")


[5] Recall@10 (HNSW vs exacta): 60.00%

================ RESUMEN QDRANT ================
Carga 20 frases:        0.015 s
K=3  aciertos 4/5 | tiempo min/max/prom: 4.8 / 36.4 / 18.0 ms
K=5  aciertos 4/5 | tiempo min/max/prom: 6.0 / 31.7 / 19.3 ms
K=10 aciertos 5/5 | tiempo min/max/prom: 5.1 / 26.4 / 15.4 ms
Carga 20000 vectores: 4.91 s (+ índice 3.91 s)
Latencia 20000 (prom): 17.8 ms | p95 32.0 ms
Recall@10:              60.00%

Guardado en C:\Users\USER\Proyectos\proyecto_BDII\resultados_qdrant.txt
